# pangram

> Score prose with Pangram's AI detector, as an alternative to the rules

In [ ]:
#| default_exp pangram

In [ ]:
#| hide
from nbdev.showdoc import *

`pangram_text` and `pangram_path` send prose to [Pangram](https://www.pangram.com)'s AI-detection API instead of running the rules. They read the same Markdown as `score_text` and `score_path`. They return a `Result` with the same addresses, JSON fields and `min_words` cutoff. `slopometer --pangram` runs `pangram_path` from the command line.

Pangram scores text in windows, which can run to hundreds of words. Each finding is one window. Its weight is the window's score multiplied by 10. A window that Pangram rates fully AI-written weighs 10, the same as a kill-tier rule finding. The density is the mean window score, weighted by word count, on a scale from 0 (human) to 100 (AI). Under `--pangram`, `--threshold` compares against this scale.

Each request sends the text to Pangram's servers and costs about $0.05 per 100 words. The functions read the API key from `PANGRAM_API_KEY`.

In [ ]:
#| export
import time, httpx2
from bisect import bisect_right
from fastcore.utils import *
from slopometer.core import *
from slopometer.segment import *
from slopometer.score import Result, _score_file, _doc_paths

In [ ]:
import json
from fastcore.test import *
from nbdev.config import get_config

## What Pangram reads

Pangram reads the blocks the rules read: headings, list items and paragraphs, with code removed. `_pangram_input` joins them with single newlines and returns each block's offset in the joined text. Pangram collapses each blank line to one newline before it reports window offsets. With single newlines, its offsets match the text sent.

In [ ]:
#| export
def _pangram_input(blocks):
    "Text to send to Pangram for `blocks`, and each block's offset in it"
    offs, n = [], 0
    for b in blocks:
        offs.append(n)
        n += len(b.txt) + 1
    return '\n'.join(b.txt for b in blocks), offs

In [ ]:
sd = get_config().config_path/'samples'
t2 = (sd/'theory2.md').read_text()
sent, offs = _pangram_input(segment(t2))
for b,o in zip(segment(t2), offs): test_eq(sent[o:o+len(b.txt)], b.txt)
sent[:200]

## Calling Pangram

Pangram runs each request as a task. `_predict` posts the text to `/task`, then polls `/task/{id}` every half second until the task's `stage` is `STAGE_SUCCESS` or `STAGE_FAILED`. It returns Pangram's response as a dict. It raises an error when the task fails or takes longer than `timeout` seconds.

In [ ]:
#| export
PANGRAM_URL = 'https://text.external-api.pangram.com'

def _predict(
    text, # Text to classify
    timeout=120, # Seconds to wait for the task to finish
):
    "Pangram's response for `text`, from a task posted to its API and polled until done"
    with httpx2.Client(base_url=PANGRAM_URL, headers={'x-api-key': os.environ['PANGRAM_API_KEY']}) as cli:
        tid = cli.post('/task', json=dict(text=text, model='default')).raise_for_status().json()['task_id']
        for _ in range(timeout*2):
            res = cli.get(f'/task/{tid}').raise_for_status().json()
            if res['stage'] == 'STAGE_SUCCESS': return res
            if res['stage'] == 'STAGE_FAILED': raise RuntimeError(f'Pangram task {tid} failed')
            time.sleep(0.5)
    raise TimeoutError(f'Pangram task {tid} did not finish in {timeout} seconds')

`samples/theory2.pangram.json` holds Pangram's response for `samples/theory2.md`. The examples below read it, so they run without an API key or network access. This cell recorded it:

In [ ]:
#| eval: false
(sd/'theory2.pangram.json').write_text(json.dumps(_predict(sent), indent=1))

In [ ]:
payload = json.loads((sd/'theory2.pangram.json').read_text())
test_eq(payload['text'], sent)
{k: payload[k] for k in ('prediction_short', 'fraction_ai', 'fraction_ai_assisted', 'fraction_human')}, len(payload['windows'])

## Windows as findings

A `PangramFinding` is a `Finding` for one window. It adds Pangram's score, label, confidence and word count. Its display shows only the start of the window, because a window can run to hundreds of words.

In [ ]:
#| export
class PangramFinding(Finding):
    def __init__(self,
        start, # Character offset where the window starts
        end, # Character offset one past the window's end
        text, # The window's text
        score, # Pangram's `ai_assistance_score`, from 0 (human) to 1 (AI)
        label, # Pangram's label, such as 'AI-Generated' or 'Human Written'
        confidence, # 'High', 'Medium', or 'Low'
        words, # Words in the window
    ):
        super().__init__('pangram', None, start, end, text, round(10*score))
        store_attr('score,label,confidence,words')
    def __repr__(self):
        ex = self.text if len(self.text) <= 80 else self.text[:77]+'...'
        return f'[{self.weight}] pangram {self.score:.2f} {self.label}, {self.confidence} confidence, {self.words} words: {ex!r}'

In [ ]:
w = payload['windows'][0]
pf = PangramFinding(w['start_index'], w['end_index'], w['text'], w['ai_assistance_score'], w['label'], w['confidence'], w['word_count'])
test_eq(pf.weight, 10)
pf

`PangramResult` keeps Pangram's response in `payload`. Its header shows the Pangram score in place of the rules' total weight.

In [ ]:
#| export
class PangramResult(Result):
    def __init__(self,
        txt, # The scored document
        findings, # `PangramFinding`s, worst first
        words, # Word count across all scored blocks
        payload=None, # Pangram's response; None when no request was sent
        min_words=150, # Minimum scored words; 0 disables the cutoff
    ):
        super().__init__(txt, findings, words, min_words=min_words)
        self.payload = payload
    def _header(self): return f'pangram score {self.density} on {self.words} prose words, worst {self.worst}'

In [ ]:
#| export
@patch(as_prop=True)
def density(self:PangramResult):
    "Mean window score, weighted by word count and scaled to 0-100"
    if self.too_short: return None
    n = sum(f.words for f in self.findings)
    return round(100*sum(f.score*f.words for f in self.findings)/max(n, 1), 1)

Density is the mean window score, weighted by word count and scaled to 0-100. A 300-word window scored 1.0 and a 100-word window scored 0.0 give a score of 75:

In [ ]:
wins = [PangramFinding(0, 50, 'x'*50, 1.0, 'AI-Generated', 'High', 300), PangramFinding(50, 80, 'y'*30, 0.0, 'Human Written', 'High', 100)]
test_eq(PangramResult('x'*80, wins, 400, min_words=0).density, 75.0)

`_pangram_result` maps each window's offsets in the joined text back to the document. It raises `ValueError` if Pangram's text differs from the text sent. Offsets into changed text would point at the wrong place. The recorded response has one window, covering the whole sample:

In [ ]:
#| export
def _pangram_result(txt, payload, min_words=150):
    "`PangramResult` for markdown `txt`, with the windows in Pangram's `payload` mapped back to document offsets"
    blocks = segment(txt)
    sent, offs = _pangram_input(blocks)
    if payload['text'] != sent: raise ValueError('Pangram scored different text from the text sent, and its window offsets cannot be mapped')
    def doc_off(i):
        k = bisect_right(offs, i) - 1
        return blocks[k].start + min(i-offs[k], len(blocks[k].txt))
    fs = [PangramFinding(doc_off(w['start_index']), doc_off(w['end_index']-1)+1, w['text'], w['ai_assistance_score'],
        w['label'], w['confidence'], w['word_count']) for w in payload['windows']]
    return PangramResult(txt, sorted(fs, key=lambda f: (-f.score, f.start)), scored_words(blocks), payload, min_words)

In [ ]:
r2 = _pangram_result(t2, payload, min_words=0)
f = r2.findings[0]
test_eq(t2[f.start:f.start+40], f.text[:40])
test_eq(t2[f.end-40:f.end], f.text[-40:])
r2

## Scoring text and files

`pangram_text`, `pangram_path` and `pangram_paths` take the same arguments as `score_text`, `score_path` and `score_paths`.

In [ ]:
#| export
def pangram_text(txt, min_words=150):
    "Score markdown `txt` with Pangram, sending no request below `min_words` scored words"
    if min_words < 0: raise ValueError('min_words must be non-negative')
    blocks = segment(txt)
    words = scored_words(blocks)
    if not words or words < min_words: return PangramResult(txt, [], words, min_words=min_words)
    return _pangram_result(txt, _predict(_pangram_input(blocks)[0]), min_words)

Below `min_words` scored words, `pangram_text` returns a short result without sending a request:

In [ ]:
short = pangram_text('robust widgets')
assert short.too_short
test_eq(short.density, None)

In [ ]:
#| export
def pangram_path(p, min_words=150):
    "Score a Markdown file, an `.ipynb` file's Markdown cells, or a `.py` file's module docstring with Pangram, with exhash addresses"
    return _score_file(p, pangram_text, min_words)

def pangram_paths(paths, n_workers=8, min_words=150):
    "Score files and directories with Pangram in parallel threads, one `PangramResult` per file"
    return parallel(pangram_path, _doc_paths(paths), n_workers=n_workers, threadpool=True, min_words=min_words)

With `PANGRAM_API_KEY` set, score a file:

In [ ]:
#| eval: false
pangram_path(sd/'theory2.md', min_words=0)

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()